# Solution Key — Functional Programming
## `filter` Lab & Partials Lab

Self-contained — the file-based part writes its own sample log first. Each part has a verified sample run and a short grading note.

---
## Lab 1 — `filter`

**Part A:** use `filter()` to keep the words that begin with a vowel.

In [ ]:
words = ['apple', 'banana', 'egg', 'fig', 'orange',
         'kiwi', 'umbrella', 'pear', 'igloo']

vowels = 'aeiouAEIOU'   # include upper-case so 'Apple' would match too

vowel_words = list(filter(lambda w: w[0] in vowels, words))
print(vowel_words)

> **Note:** the predicate is `lambda w: w[0] in vowels`. Watch for two things: handling an empty string (`w[0]` would raise `IndexError` — add `w and ...` if robustness matters), and case (a vowel check on `w[0].lower()` is a clean alternative to listing both cases).

**Part B:** extend the deque-`tail` from the collections lab so it shows only lines matching a string or a regex. Because this is the `filter` lab, the natural move is to `filter()` the file *before* feeding it to the deque — so the deque keeps the last *n* **matching** lines (like `grep ... | tail`).

In [ ]:
from collections import deque
import re

# Write a small sample log.
with open('app.log', 'w') as f:
    f.write(
        '01 INFO boot\n'
        '02 error code 500 timeout\n'
        '03 INFO ok\n'
        '04 error code 512 retry\n'
        '05 INFO salesforce sync start\n'
        '06 error code 418 teapot\n'
        '07 WARN salesforce slow\n'
        '08 error code 502 gateway\n'
        '09 INFO done\n'
        '10 error code 511 odd\n'
    )


def tail_matching(filename, pattern, n=10, is_regex=False):
    """Print the last n lines that match `pattern` (substring or regex)."""
    if is_regex:
        matches = lambda line: re.search(pattern, line)
    else:
        matches = lambda line: pattern in line
    with open(filename) as f:
        last = deque(filter(matches, f), maxlen=n)   # filter THEN keep last n
    for line in last:
        print(line, end='')


print("--- contains 'salesforce' ---")
tail_matching('app.log', 'salesforce')
print("\n--- regex 'error.*5[012]', last 3 ---")
tail_matching('app.log', 'error.*5[012]', n=3, is_regex=True)

> **Key point:** wrapping the file in `filter(matches, f)` and then `deque(..., maxlen=n)` streams the file and only ever holds *n* lines — it scales to huge logs. Either filter order is defensible (filter-then-tail vs. tail-then-filter). The regex `error.*5[012]` matches codes 50x/51x/52x — here 512, 502, 511.

---
## Lab 2 — Partials

`functools.partial` freezes some arguments of a function to make a simpler one. Each of these is a one-liner — that *is* the point of the lab.

In [ ]:
from functools import partial

# Freeze the keyword arguments that you'd otherwise have to type every time.
print_no_nl = partial(print, end='')    # print without a trailing newline
print_no_sp = partial(print, sep='')    # print without separators between args
sorted_r = partial(sorted, reverse=True)  # sort in reverse order

# print_no_nl: the two calls land on the same line
print_no_nl('hello')
print_no_nl(' world')
print()   # finish the line

# print_no_sp: no spaces between the arguments
print_no_sp('a', 'b', 'c')

# sorted_r: reverse sorts
print(sorted_r([3, 1, 4, 1, 5, 9, 2]))
print(sorted_r(['banana', 'apple', 'cherry']))

> **Note:** each should be a `partial` that freezes a keyword argument (`end=''`, `sep=''`, `reverse=True`) — not a hand-written wrapper `def` (which works but misses the lesson). A subtle win of `sorted_r = partial(sorted, reverse=True)` is that it still accepts `key=...` at call time, because `partial` only pins the arguments you gave it.